# Automated Layer Search

Train probes on all layers with different pooling strategies to find where negation is represented.


In [ ]:
import sys
import os
from pathlib import Path

# Ensure we're in the right directory
if not os.path.exists('src'):
    if os.path.exists('Negation-Origin-Tracing'):
        os.chdir('Negation-Origin-Tracing')
    else:
        print("⚠ Warning: Could not find project directory")
        print(f"  Current: {os.getcwd()}")
        print(f"  Run 00_setup_colab.ipynb first!")

sys.path.insert(0, os.getcwd())
print(f"✓ Working directory: {os.getcwd()}")


In [ ]:
# Configuration
import torch

# Auto-detect GPU
use_gpu = torch.cuda.is_available()

config = {
    'data_dir': 'data/raw',
    'model_name': 'distilbert-base-uncased',
    'batch_size': 32 if use_gpu else 16,  # Smaller batch if no GPU
    'max_epochs': 10,
    'probe_lr': 1e-3,
    'seed': 42,
    'layers': 'all',  # or comma-separated like '0,1,2,3,4,5'
    'pooling_strategies': 'all',  # or 'cls,mean,token'
    'output_dir': 'experiments/layer_search',
    'devices': 1 if use_gpu else 1,  # Use GPU if available
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

if use_gpu:
    print(f"\n✓ Will use GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠ No GPU - training will be slower. Consider enabling GPU in Colab.")


In [ ]:
# Run layer search using the script
import subprocess

cmd = [
    'python', 'src/scripts/search_layers.py',
    '--data_dir', config['data_dir'],
    '--model_name', config['model_name'],
    '--batch_size', str(config['batch_size']),
    '--max_epochs', str(config['max_epochs']),
    '--probe_lr', str(config['probe_lr']),
    '--output_dir', config['output_dir'],
    '--layers', config['layers'],
    '--pooling_strategies', config['pooling_strategies'],
    '--seed', str(config['seed']),
    '--devices', str(config['devices']),
]

print("Running layer search...")
print(f"Command: {' '.join(cmd)}")
result = subprocess.run(cmd, check=True)
print("\n✓ Layer search complete!")


In [ ]:
# Load and display results
import pandas as pd
import json

results_path = os.path.join(config['output_dir'], 'results_summary.json')

if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        results = json.load(f)
    
    df = pd.DataFrame(results)
    
    print("\nResults Summary:")
    print(f"Total experiments: {len(df)}")
    print(f"\nBest Test Accuracy:")
    best_acc = df.loc[df['test_accuracy'].idxmax()]
    print(f"  Layer {best_acc['layer_idx']}, {best_acc['pooling_strategy']}: {best_acc['test_accuracy']:.4f}")
    
    print(f"\nBest Test AUROC:")
    best_auroc = df.loc[df['test_auroc'].idxmax()]
    print(f"  Layer {best_auroc['layer_idx']}, {best_auroc['pooling_strategy']}: {best_auroc['test_auroc']:.4f}")
    
    print("\nAverage Performance by Layer:")
    print(df.groupby('layer_idx')[['test_accuracy', 'test_auroc']].mean().round(4))
    
    print("\nAverage Performance by Pooling Strategy:")
    print(df.groupby('pooling_strategy')[['test_accuracy', 'test_auroc']].mean().round(4))
else:
    print(f"⚠ Results file not found at {results_path}")
